# BioPrimeLASSO Python Demo
This notebook fabricates a small synthetic dataset to demonstrate how the Python implementation orchestrates score generation, hyperparameter tuning, fitting, prediction, and plotting.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from bioprimelasso.backend import SklearnLassoBackend
from bioprimelasso.model import BioPrimeLassoModel
from bioprimelasso.repository import LocalResultRepository
from bioprimelasso.scores import NetworkScoreProvider
from bioprimelasso.tuning import DefaultHyperparameterTuner
from bioprimelasso.plotting import PlotRegistry


In [ ]:
rng = np.random.default_rng(42)
n_samples, n_features = 60, 25
feature_names = [f"Gene_{i:03d}" for i in range(n_features)]
cell_lines = [f"CL_{i:03d}" for i in range(n_samples)]

X = pd.DataFrame(rng.normal(size=(n_samples, n_features)), index=cell_lines, columns=feature_names)
true_beta = np.zeros(n_features)
true_beta[[2, 5, 11]] = [1.0, -1.5, 0.8]
y = pd.Series(X.values @ true_beta + rng.normal(scale=0.3, size=n_samples), index=cell_lines, name="dependency")

network = pd.DataFrame({
    "gene1": ["GeneOfInterest"] * n_features,
    "gene2": feature_names,
    "combined_score": np.clip(rng.uniform(0.2, 1.0, size=n_features) + (true_beta != 0) * 0.4, 0, 1.0),
})


In [ ]:
score_provider = NetworkScoreProvider()
tuner = DefaultHyperparameterTuner()
backend = SklearnLassoBackend()
repository = LocalResultRepository(Path("./demo_results"))
plotters = PlotRegistry.default()

model = BioPrimeLassoModel(
    tuner=tuner,
    backend=backend,
    result_store=repository,
    score_provider=score_provider,
    plotters=plotters,
)


In [ ]:
handle = model.fit(
    X,
    y,
    gene="GeneOfInterest",
    network=network,
    metadata={"description": "Synthetic demonstration dataset"},
)
handle


In [ ]:
cv_report = model.cross_validate(
    X,
    y,
    folds=5,
    gene="GeneOfInterest",
    network=network,
)
cv_report.best_phi, cv_report.rmse.mean(axis=1).head()


In [ ]:
stored = repository.load(handle)
stored.baseline_betas.head()


In [ ]:
output_path = Path("./demo_manhattan.pdf")
model.plot_manhattan(handle, output=output_path)
output_path
